# Sentiment classification with two text representations





We use a dataset of app reviews labeled with sentiment (0 = negative, 1 = neutral, 2 = positive).

In [2]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 20.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import pandas as pd
import numpy as np
import re
import spacy
import tensorflow as tf

nlp = spacy.load('en_core_web_md')
tf.random.set_seed(42)
from google.colab import files

uploaded = files.upload()
path_to_file = 'all_data.csv'
data = pd.read_csv(path_to_file)
data = data.dropna(subset=['review', 'sentiment']).reset_index(drop=True)
len(data)

Saving all_data.csv to all_data.csv


40516

In [5]:
# a quick look at a raw example before cleaning
en_review = data['review'].iloc[1]
print(en_review)

I love the app.! There is no issue but if u could add the feature of do not spotlight on unmute in Android devices then it would be more nice


### Clean the text

We remove non-alphabetic characters, extra whitespace, and lowercase everything, exactly the same way for both representations we're about to build.

In [6]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data['clean_text'] = data['review'].map(preprocess_text)
data = data[data['clean_text'].str.len() > 0].reset_index(drop=True)
len(data)

39804

### Creating training and validation sets

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data['clean_text'], data['sentiment'], test_size=0.2, random_state=42, stratify=data['sentiment']
)
y_train_arr = np.array(y_train)
y_test_arr = np.array(y_test)
len(X_train), len(X_test)

(31843, 7961)

## Write the two representation extractors

Just like an encoder turns raw input into a fixed-size representation the rest of the model can work with, here we write two small extractor classes: one turns text into a TF-IDF vector, the other turns text into a spaCy embedding vector.

In [8]:
class TfidfExtractor:
    def __init__(self, max_features=5000):
        from sklearn.feature_extraction.text import TfidfVectorizer
        self.vectorizer = TfidfVectorizer(max_features=max_features)

    def fit_transform(self, texts):
        return self.vectorizer.fit_transform(texts).toarray().astype('float32')

    def transform(self, texts):
        return self.vectorizer.transform(texts).toarray().astype('float32')


class SpacyExtractor:
    def __init__(self, nlp_model):
        self.nlp = nlp_model

    def fit_transform(self, texts):
        return self.transform(texts)

    def transform(self, texts):
        return np.array([doc.vector for doc in self.nlp.pipe(texts, batch_size=256)], dtype='float32')

In [9]:
tfidf_extractor = TfidfExtractor(max_features=5000)
X_train_tfidf = tfidf_extractor.fit_transform(X_train)
X_test_tfidf = tfidf_extractor.transform(X_test)

spacy_extractor = SpacyExtractor(nlp)
X_train_emb = spacy_extractor.fit_transform(X_train)
X_test_emb = spacy_extractor.transform(X_test)

print(X_train_tfidf.shape, X_train_emb.shape)

(31843, 5000) (31843, 300)


## Define the classifier and the training procedure


In [10]:
def build_classifier(input_dim, num_classes=3):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [11]:
EPOCHS = 8

clf_tfidf = build_classifier(input_dim=X_train_tfidf.shape[1])
clf_tfidf.fit(X_train_tfidf, y_train_arr, epochs=EPOCHS, batch_size=128, validation_split=0.1, verbose=0)

clf_emb = build_classifier(input_dim=X_train_emb.shape[1])
clf_emb.fit(X_train_emb, y_train_arr, epochs=EPOCHS, batch_size=128, validation_split=0.1, verbose=0)
print("Training complete for both representations.")

Training complete for both representations.


## Evaluate


In [12]:
from sklearn.metrics import f1_score, classification_report

test_loss_tfidf, test_acc_tfidf = clf_tfidf.evaluate(X_test_tfidf, y_test_arr, verbose=0)
pred_tfidf = np.argmax(clf_tfidf.predict(X_test_tfidf, verbose=0), axis=1)
f1_tfidf = f1_score(y_test_arr, pred_tfidf, average='macro')
print(f"TF-IDF -> accuracy: {test_acc_tfidf:.4f}, macro F1: {f1_tfidf:.4f}")
print(classification_report(y_test_arr, pred_tfidf))

TF-IDF -> accuracy: 0.6901, macro F1: 0.6757
              precision    recall  f1-score   support

           0       0.72      0.72      0.72      2657
           1       0.63      0.50      0.56      2302
           2       0.70      0.81      0.75      3002

    accuracy                           0.69      7961
   macro avg       0.68      0.68      0.68      7961
weighted avg       0.69      0.69      0.68      7961



In [13]:
test_loss_emb, test_acc_emb = clf_emb.evaluate(X_test_emb, y_test_arr, verbose=0)
pred_emb = np.argmax(clf_emb.predict(X_test_emb, verbose=0), axis=1)
f1_emb = f1_score(y_test_arr, pred_emb, average='macro')
print(f"spaCy embeddings -> accuracy: {test_acc_emb:.4f}, macro F1: {f1_emb:.4f}")
print(classification_report(y_test_arr, pred_emb))

spaCy embeddings -> accuracy: 0.5653, macro F1: 0.5273
              precision    recall  f1-score   support

           0       0.55      0.67      0.61      2657
           1       0.47      0.24      0.32      2302
           2       0.61      0.72      0.66      3002

    accuracy                           0.57      7961
   macro avg       0.54      0.54      0.53      7961
weighted avg       0.55      0.57      0.54      7961



## Test on new examples

In [14]:
def predict_sentiment(text):
    clean = preprocess_text(text)
    mapping = {0: 'negative', 1: 'neutral', 2: 'positive'}
    tfidf_pred = mapping[np.argmax(clf_tfidf.predict(tfidf_extractor.transform([clean]), verbose=0), axis=1)[0]]
    emb_pred = mapping[np.argmax(clf_emb.predict(spacy_extractor.transform([clean]), verbose=0), axis=1)[0]]
    return {'tfidf': tfidf_pred, 'spacy': emb_pred}

predict_sentiment('excellent app, highly recommend it to everyone')

{'tfidf': 'positive', 'spacy': 'positive'}

In [15]:
predict_sentiment('cannot log in anymore, completely useless now')

{'tfidf': 'negative', 'spacy': 'negative'}

In [16]:
# an example where the two representations might disagree
predict_sentiment('it works i guess, nothing special')

{'tfidf': 'neutral', 'spacy': 'positive'}

## Next steps

* Try different `max_features` values for TF-IDF, or a different spaCy model (e.g. `en_core_web_lg`) for the
  embeddings, to see how sensitive each representation is to these choices.
* This notebook confirms, on a third framing of the same experiment (all built with `tf.keras`), which
  representation held up better under identical preprocessing and the identical classifier, for this specific
  noisy, multilingual app-review dataset.